In [0]:
%python
BOOTSTRAP_SERVERS = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
API_KEY = "KVSN226VZUCGXMIE"
API_SECRET = "cfltd8VtX2TS0PpyY58UcLetDr6KlxFQQMQna9fI1gjwbJ0Lo4vLgBejm2U4w2Gg"
TOPIC_NAME = "sample_data_orders"

In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
display(spark.sql("SHOW VOLUMES"))

database,volume_name
default,azuredatabricks_catalog


In [0]:
display(spark.sql("SELECT current_catalog()"))

current_catalog()
workspace


In [0]:
one_batch_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

display(one_batch_df.selectExpr("CAST(value AS STRING) as raw_value").limit(1))

raw_value
"  ��{""ordertime"":1512838934228,""orderid"":4,""itemid"":""Item_18"",""orderunits"":2.5190939004768587,""address"":{""city"":""City_"",""state"":""State_66"",""zipcode"":24513}}"


In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

dbutils.fs.rm("/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/raw_stream_live", recurse=True)

display( raw_stream_df.selectExpr("CAST(value AS STRING) as raw_value"), checkpointLocation="/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/raw_stream_live" )

Checkpointing to /Volumes/workspace/default/azuredatabricks_catalog/checkpoints/raw_stream_live


In [0]:
from pyspark.sql.types import StructType, StringType, LongType, IntegerType, DoubleType, StructField

order_schema = (
    StructType()
    .add("ordertime", LongType())
    .add("orderid", IntegerType())
    .add("itemid", StringType())
    .add("orderunits", DoubleType())
    .add("address", StructType()
         .add("city", StringType())
         .add("state", StringType())
         .add("zipcode", LongType())
    )
)

In [0]:
from pyspark.sql.functions import from_json, col

parsed_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_value")
    .select(from_json(col("json_value"), order_schema).alias("data"))
    .select("data.*")
)

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v2", recurse=True)

display(
    parsed_df,
    checkpointLocation="/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v2"
)

Checkpointing to /Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v2


In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

from pyspark.sql.functions import from_json, col

parsed_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_value")
    .select(from_json(col("json_value"), order_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm("/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v3", recurse=True)

display(
    parsed_df,
    checkpointLocation="/Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v3"
)

Checkpointing to /Volumes/workspace/default/azuredatabricks_catalog/checkpoints/parsed_live_v3
